## 02 — Viewport Culling on a Live Map

We have `bbox_intersects`. Now we connect it to an actual map viewport.

ipyleaflet exposes the map's current bounds via `m.bounds` — the visible geographic rectangle. We read that rectangle, run the intersection test against every feature in the active LOD file, and send only the survivors to the renderer.

This notebook:
1. Reads the viewport from an ipyleaflet map
2. Applies the cull filter to a LOD file
3. Measures feature reduction at different zoom levels
4. Renders the culled result live

## Setup

In [1]:
import json
from pathlib import Path

lod_dir = Path("../../data/lod")

# Load the fine LOD file — a good test since it has the most features
with open(lod_dir / "railroads_fine.geojson") as f:
    fine = json.load(f)

features = fine["features"]
print(f"Fine LOD: {len(features):,} features")

Fine LOD: 25,413 features


In [2]:
def feature_bbox(feature):
    coords = feature["geometry"]["coordinates"]
    lons = [c[0] for c in coords]
    lats = [c[1] for c in coords]
    return [min(lons), min(lats), max(lons), max(lats)]


def bbox_intersects(bbox_a, bbox_b):
    a_lon_min, a_lat_min, a_lon_max, a_lat_max = bbox_a
    b_lon_min, b_lat_min, b_lon_max, b_lat_max = bbox_b
    if a_lon_max < b_lon_min: return False
    if a_lon_min > b_lon_max: return False
    if a_lat_max < b_lat_min: return False
    if a_lat_min > b_lat_max: return False
    return True


def cull(features, viewport_bbox):
    """Return only the features whose bbox intersects viewport_bbox."""
    return [
        f for f in features
        if bbox_intersects(feature_bbox(f), viewport_bbox)
    ]

## Reading the Viewport from ipyleaflet

ipyleaflet's `m.bounds` returns the current visible bounds as:

```python
[[lat_min, lon_min], [lat_max, lon_max]]
```

Note the order: **latitude first, longitude second**. This is ipyleaflet's convention — the opposite of GeoJSON's `[lon, lat]`.

We need a helper to convert from ipyleaflet bounds to our `[lon_min, lat_min, lon_max, lat_max]` format.

In [3]:
def leaflet_bounds_to_bbox(bounds):
    """
    Convert ipyleaflet m.bounds format  [[lat_min, lon_min], [lat_max, lon_max]]
    to our bbox format                  [lon_min, lat_min, lon_max, lat_max]
    """
    (lat_min, lon_min), (lat_max, lon_max) = bounds
    return [lon_min, lat_min, lon_max, lat_max]

## Static Culling — Measuring Reduction

Before building the live map, let's measure how effective culling is at several representative viewports.

In [4]:
viewports = [
    ("World (zoom 2)",          [-180, -90,  180,  90]),
    ("Europe (zoom 5)",         [ -10,  35,   40,  70]),
    ("France (zoom 7)",         [  -5,  42,   10,  52]),
    ("Paris region (zoom 10)",  [   1,  48,    4,  50]),
    ("Central Paris (zoom 13)", [ 2.3, 48.8, 2.4, 48.9]),
]

total = len(features)

print(f"{'Viewport':<26} {'Visible':>8} {'Culled':>8} {'% kept':>8}")
print("-" * 55)

for name, vp in viewports:
    visible = cull(features, vp)
    pct = len(visible) / total * 100
    culled = total - len(visible)
    print(f"{name:<26} {len(visible):>8,} {culled:>8,} {pct:>7.1f}%")

Viewport                    Visible   Culled   % kept
-------------------------------------------------------
World (zoom 2)               25,413        0   100.0%
Europe (zoom 5)              12,626   12,787    49.7%
France (zoom 7)               1,598   23,815     6.3%
Paris region (zoom 10)           51   25,362     0.2%
Central Paris (zoom 13)          11   25,402     0.0%


At city-level zoom, culling eliminates almost everything. At world zoom, nothing is culled — which is expected and acceptable because we would use the coarse LOD at that zoom anyway.

## Live Map with Viewport Culling

Now we wire it together. When the user stops panning or zooming, we:
1. Read `m.bounds`
2. Convert to our bbox format
3. Cull the features
4. Replace the GeoJSON layer with the culled result

In [5]:
from ipyleaflet import Map, GeoJSON
import ipywidgets as widgets

m = Map(center=[48.5, 2.5], zoom=6)
status = widgets.Label(value="Pan or zoom to trigger culling")

layer = GeoJSON(
    data={"type": "FeatureCollection", "features": []},
    style={"color": "#cc3300", "weight": 1.5, "opacity": 0.8}
)
m.add(layer)

def update_layer(*args):
    if not m.bounds:
        return
    viewport_bbox = leaflet_bounds_to_bbox(m.bounds)
    visible = cull(features, viewport_bbox)
    layer.data = {"type": "FeatureCollection", "features": visible}
    status.value = f"Viewport: {[round(v,2) for v in viewport_bbox]}  |  {len(visible):,} / {len(features):,} features visible"

m.observe(update_layer, names=["bounds"])
update_layer()  # Run once on load

widgets.VBox([m, status])

Pan around and zoom in. Watch the status line — the feature count drops dramatically as you zoom into a small region.

Notice also a limitation: each pan triggers a full linear scan of all features to recompute culling. With 25,000 features this is fast enough to feel acceptable. At 250,000 features it would lag. This is the motivation for Module 04 — a spatial grid index.

## Exercise A

The current map always uses the `fine` LOD file regardless of zoom. Modify the map so it uses `railroads_coarse.geojson` when zoom < 4 and `railroads_fine.geojson` when zoom >= 4.

This is a preview of Module 05 — just get it working here as a one-off.

In [7]:
# Load both LOD files and switch between them based on zoom level
# Your code here

with open(lod_dir / "railroads_coarse.geojson") as f:
    coarse = json.load(f)

def get_lod_features(zoom):
    return coarse["features"] if zoom < 4 else fine["features"]

def update_layer(*args):
    if not m.bounds:
        return

    viewport_bbox = leaflet_bounds_to_bbox(m.bounds)
    active_features = get_lod_features(m.zoom)

    visible = cull(active_features, viewport_bbox)

    layer.data = {
        "type": "FeatureCollection",
        "features": visible
    }

    status.value = f"zoom={m.zoom:.1f} | {len(visible):,}/{len(active_features):,} visible"

m.observe(update_layer, names=["bounds", "zoom"])
update_layer()

m

Map(bottom=90439.99998291016, center=[48.806863476542055, 2.6360333335218256], controls=(ZoomControl(options=[…

## Exercise B

The culling function runs a **linear scan** — it checks every feature in order. For the fine LOD with ~20,000 features, measure how long a single cull call takes using `time.perf_counter()`.

Then calculate: if the user pans 10 times per second, how much CPU time does culling consume per second? Is this a problem?

In [ ]:
import time

# Time a single cull() call on the fine LOD
# Calculate CPU cost at 10 pans/second
# Your code here

viewport = (-10, 35, 30, 60)

t0 = time.perf_counter()
visible = cull(features, viewport)
t1 = time.perf_counter()

single_time = t1 - t0

print(f"Single cull time: {single_time:.6f} seconds")
print(f"Visible features: {len(visible):,}")
print(f"Total features: {len(features):,}")

Single cull time: 0.064559 seconds
Visible features: 9,836
Total features: 25,413


In [9]:
updates_per_second = 10
cpu_per_second = single_time * updates_per_second

print(f"CPU time per second: {cpu_per_second:.4f} seconds")
print(f"CPU utilization: {cpu_per_second * 100:.2f}% of one core")

CPU time per second: 0.6456 seconds
CPU utilization: 64.56% of one core


## Check Your Understanding

Our culling test uses **bounding boxes of the features**, not the actual line geometry.

A long diagonal railroad that runs from the bottom-left to the top-right corner of Europe has a bounding box that covers the entire continent. When the user is zoomed into Portugal, this feature passes the cull test — even though the actual track is nowhere near Portugal.

Is this a flaw in the algorithm? What would a more precise solution require — and is it worth it for a LOD rendering pipeline?

---

As I mentioned in the last micro lesson notebook, this would be fine as a more precise solution would require checking the coordinates of every single point on every single feature to see if they were within the viewport. This introduces a lot of overhead and extra checking time, so I don't think it's worth it.

## Next

In [Module 04 — Spatial Grid Index](../04-Spatial_Grid_Index/README.md), we replace the linear scan with a grid-based index that makes culling faster as the dataset grows.